# 🧮 Generation Metrics

## What We're Measuring

Once the LLM generates an answer, we need to evaluate it.

There are two dimensions:

```
1. Faithfulness      — Does the answer stick to the retrieved docs?
                       (No hallucination)

2. Answer Relevance  — Does the answer actually address the question?
                       (No off-topic responses)
```

A good RAG answer scores high on **both**.

## Metrics We'll Cover

| Metric | What it checks |
|---|---|
| **Exact Match** | Does the answer contain the ground truth string? |
| **Token Overlap (F1)** | Word-level overlap with the reference answer |
| **Semantic Similarity** | Embedding-based similarity to reference |
| **Faithfulness Score** | What % of answer claims are supported by context? |

In [ ]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Ready!")

In [ ]:
# Sample evaluation data
eval_samples = [
    {
        "question": "What is dropout?",
        "context": "Dropout randomly deactivates neurons during training to prevent overfitting.",
        "reference_answer": "Dropout randomly deactivates neurons during training to prevent overfitting.",
        "generated_answer": "Dropout is a regularization technique that randomly turns off neurons during training, helping prevent overfitting.",
    },
    {
        "question": "What is batch normalization?",
        "context": "Batch normalization normalizes layer inputs to speed up training and stabilize the network.",
        "reference_answer": "Batch normalization normalizes the inputs of each layer to stabilize and speed up training.",
        "generated_answer": "Batch normalization was invented at Google and is used in ResNets to improve performance.",  # Contains hallucination!
    },
    {
        "question": "What is transfer learning?",
        "context": "Transfer learning reuses weights from a model trained on one task for a different but related task.",
        "reference_answer": "Transfer learning reuses pretrained model weights on a new related task.",
        "generated_answer": "Transfer learning involves reusing a model's learned weights for a new task.",
    },
]

print(f"Evaluation samples: {len(eval_samples)}")

## 1. Exact Match

In [ ]:
# Exact match: normalize both strings, check if they're identical
# Simple but useful for factual questions with short answers

def normalize(text):
    """Lowercase, strip punctuation and extra spaces."""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def exact_match(prediction, reference):
    return int(normalize(prediction) == normalize(reference))


print("Exact Match:")
for s in eval_samples:
    score = exact_match(s["generated_answer"], s["reference_answer"])
    print(f"  {score} | Q: {s['question']}")

print("\n💡 Exact match is strict — paraphrases still count as wrong.")

## 2. Token F1 (Word Overlap)

In [ ]:
# Token F1: measures word-level overlap between prediction and reference
# Used in SQuAD benchmark — much more forgiving than exact match

def token_f1(prediction, reference):
    pred_tokens = set(normalize(prediction).split())
    ref_tokens  = set(normalize(reference).split())

    if not pred_tokens or not ref_tokens:
        return 0.0

    common = pred_tokens & ref_tokens
    precision = len(common) / len(pred_tokens)
    recall    = len(common) / len(ref_tokens)

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


print("Token F1 (word overlap):")
for s in eval_samples:
    score = token_f1(s["generated_answer"], s["reference_answer"])
    print(f"  {score:.3f} | Q: {s['question']}")

print("\n💡 Token F1 handles paraphrasing better than exact match.")

## 3. Semantic Similarity

In [ ]:
# Semantic similarity: embed both answers, measure cosine similarity
# Best for paraphrases that share meaning but use different words

def semantic_similarity(prediction, reference):
    pred_emb = model.encode(prediction)
    ref_emb  = model.encode(reference)
    score = cosine_similarity([pred_emb], [ref_emb])[0][0]
    return float(score)


print("Semantic Similarity:")
for s in eval_samples:
    score = semantic_similarity(s["generated_answer"], s["reference_answer"])
    print(f"  {score:.3f} | Q: {s['question']}")

print("\n💡 Semantic similarity catches good paraphrases — but also misses hallucinations.")

## 4. Faithfulness Score

This is the most important metric for RAG. It asks:  
**Is every claim in the answer supported by the retrieved context?**

In [ ]:
# Faithfulness check:
# Split the answer into sentences.
# For each sentence, check if it's semantically similar to any sentence in the context.
# Sentences with no match are potential hallucinations.

FAITHFULNESS_THRESHOLD = 0.5  # Minimum similarity to count as supported

def faithfulness_score(generated_answer, context):
    """
    Returns a score 0-1:
    1.0 = every sentence in the answer is supported by context
    0.0 = no sentence is supported
    """
    # Split answer into sentences
    sentences = [s.strip() for s in generated_answer.split('.') if len(s.strip()) > 10]
    if not sentences:
        return 0.0

    # Split context into sentences too
    context_sentences = [s.strip() for s in context.split('.') if len(s.strip()) > 5]

    # Embed all
    sent_embs    = model.encode(sentences)
    context_embs = model.encode(context_sentences)

    # For each answer sentence, find max similarity to any context sentence
    supported = 0
    for i, sent_emb in enumerate(sent_embs):
        sims = cosine_similarity([sent_emb], context_embs)[0]
        max_sim = float(np.max(sims))
        is_supported = max_sim >= FAITHFULNESS_THRESHOLD
        if is_supported:
            supported += 1
        print(f"    {'✅' if is_supported else '⚠️ '} [{max_sim:.3f}] {sentences[i][:70]}")

    return supported / len(sentences)


# Run faithfulness check on each sample
for s in eval_samples:
    print(f"\nQuestion: {s['question']}")
    print(f"Answer: {s['generated_answer']}")
    print("Sentence-level faithfulness:")
    score = faithfulness_score(s["generated_answer"], s["context"])
    print(f"  → Faithfulness score: {score:.2f}")

## 5. All Metrics Together

In [ ]:
# Summary table
print("\nGeneration Metrics Summary")
print("=" * 70)
print(f"{'Question':<35} {'ExMatch':>8} {'TokenF1':>8} {'SemSim':>8}")
print("-" * 70)

for s in eval_samples:
    em = exact_match(s["generated_answer"], s["reference_answer"])
    f1 = token_f1(s["generated_answer"], s["reference_answer"])
    ss = semantic_similarity(s["generated_answer"], s["reference_answer"])
    print(f"{s['question']:<35} {em:>8} {f1:>8.3f} {ss:>8.3f}")

print("\n💡 Use all three — each catches different failure modes.")
print("   Faithfulness is the most important metric specifically for RAG.")

## Key Takeaways 🎯

| Metric | Catches |
|---|---|
| Exact Match | Only identical answers |
| Token F1 | Word-level paraphrases |
| Semantic Similarity | Meaning-level paraphrases |
| **Faithfulness** | **Hallucinations** ← most important for RAG |

**For production RAG:** Use [RAGAS](https://docs.ragas.io) — a library that automates all of these metrics at scale.

```python
# RAGAS usage (install with: pip install ragas)
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy

result = evaluate(dataset, metrics=[faithfulness, answer_relevancy])
print(result)
```

---
Next: `03_End_to_End_Evaluation.ipynb` — evaluate the full pipeline together.